In [ ]:
# =====================================================================
# Dynamic-Room Two-Stage Pipeline
#   Input: room dimensions (rx, ry)
#   Phase 1: NSGA-II on LGBM surrogate (200 gen, ~15s)
#   Phase 2: Physics warm-start AGE-MOEA (20 gen, ~60s)
#   Output: Pareto front for this specific room
# =====================================================================

import torch, numpy as np, math, time, json, pickle, glob, os
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import lightgbm as lgb
from pymoo.core.problem import ElementwiseProblem
from pymoo.core.population import Population
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.age import AGEMOEA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

try:
    device = torch.device('cuda'); torch.zeros(1, device=device)
except:
    device = torch.device('cpu')
print(f'Device: {device}')

# ============================================================
# 1. Load Model
# ============================================================
def find_file(pattern):
    cand = [pattern, f'/kaggle/working/{pattern}'] + sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    for c in cand:
        if os.path.exists(c): return c
    raise FileNotFoundError(pattern)

lgbm_m = lgb.Booster(model_file=find_file('lgbm_dynamic.txt'))
print('LGBM model loaded.')

# Feature engineering: 4 window params + rx,ry → 17 features
def enrich(xc,zc,Lx,Lz,rx,ry):
    xc_rel=xc/rx; Lx_rel=Lx/rx; area_ratio=(Lx*Lz)/(rx*3.0)
    dx=xc-10.0; dy=ry+100.0; dz_=zc+10.0; dist=np.sqrt(dx**2+dy**2+dz_**2)
    alpha_x=dx/dist; alpha_z=dz_/dist
    mL=xc-Lx/2; mR=rx-(xc+Lx/2); mB=zc-Lz/2; mT=3.0-(zc+Lz/2)
    aspect=Lx/(Lz+1e-6)
    return np.column_stack([xc,zc,Lx,Lz,
        np.full_like(xc,rx),np.full_like(xc,ry),
        xc_rel,Lx_rel,area_ratio,dist,alpha_x,alpha_z,
        mL,mR,mB,mT,aspect]).astype(np.float32)

def lgbm_predict(X_4d, rx, ry):
    return lgbm_m.predict(enrich(X_4d[:,0],X_4d[:,1],X_4d[:,2],X_4d[:,3],rx,ry))

# ============================================================
# 2. Physics Engine for Specific Room
# ============================================================
xBs,yBs,zBs=10.0,-100.0,-10.0; E=8; dB_s=0.075; lam=0.075
Lp=2; dref=abs(yBs)*1.5; Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6
pm=0.2; nu=5; rz=3.0; STEP=0.05
PB=10**(Pd/10)*1e-3; N0f=10**(Nd/10)*1e-3*Bw
Z_HS=[1.5,1.625,1.75,1.875,2.0]; N_Z=5

def gen_rwp(rx,ry,sim_time,dt=10):
    rng=np.random; rng.seed(777)
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=min(rx,ry)/3.0
    ts=int(sim_time/dt); p_t,p_s=0.6,0.3; tau_h,tau_n=1.5,0.1; v_h,v_n=0.2,1.0; g_h,g_n=0.6,0.2
    def gt():
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=['normal']*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])]
        d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
        else:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n);pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n)
                    else:ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
                else:up[i]=up[i]+ud[i]*md;pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

_PHYS = {}

def build_room(rx,ry):
    nx,ny=int(rx/STEP),int(ry/STEP)
    xv=torch.linspace(0,rx,nx,device=device); yv=torch.linspace(0,ry,ny,device=device)
    Xg,Yg=torch.meshgrid(xv,yv,indexing='ij')
    xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
    gl=[]
    for zh in Z_HS: gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1))
    gl=torch.cat(gl,dim=0); Ng=gl.shape[0]
    et=gen_rwp(rx,ry,100000,10)
    kde=gaussian_kde(et[:,:2].T,bw_method='scott')
    wxy=kde(xyf.cpu().numpy().T).flatten(); wxy=wxy/wxy.sum()
    gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device); gw=gw/gw.sum()
    np.random.seed(42)
    ps=torch.tensor(2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    eta=torch.tensor(10**((-15+5*np.random.rand(Ng))/10),dtype=torch.float32,device=device)
    na=torch.arange(E,dtype=torch.float32,device=device)
    dyB=torch.tensor(0.0-yBs,device=device)
    v1c=lam/(4*math.pi); v1pc=-(2*math.pi/lam); oE=1/math.sqrt(E)
    return gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE

@torch.no_grad()
def physics_outage(X_4d, rx, ry):
    key=(round(rx,2),round(ry,2))
    if key not in _PHYS: _PHYS[key]=build_room(rx,ry)
    gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE=_PHYS[key]
    out=[]
    for i in range(0,len(X_4d),200):
        b=torch.tensor(X_4d[i:i+200],dtype=torch.float32,device=device)
        bo=torch.zeros(len(b),device=device)
        for j in range(len(b)):
            xc,zc,Lx,Lz=b[j,0],b[j,1],b[j,2],b[j,3]
            xg,yg,zg=gl[:,0],gl[:,1],gl[:,2]; Ng=xg.shape[0]
            dx=xc-xBs;dy=dyB;dz=zc-zBs;Rw=torch.sqrt(dx**2+dy**2+dz**2)
            th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
            kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
            def rd(xs,zs): dd=xs-xBs;dz_=zs-zBs;L=torch.sqrt(dd**2+dy**2+dz_**2);return dd/L,dy/L,dz_/L
            x1,x2=xc-Lx/2,xc-Lx/2;x3,x4=xc+Lx/2,xc+Lx/2;z1,z3=zc-Lz/2,zc-Lz/2;z2,z4=zc+Lz/2,zc+Lz/2
            ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
            uma=torch.stack([ux1,ux2,ux3,ux4]);uzm=torch.stack([uz1,uz2,uz3,uz4])
            umin=uma.min(0).values;umax=uma.max(0).values;zmin=uzm.min(0).values;zmax=uzm.max(0).values
            dux=xg-xBs;duy=yg-yBs;duz=zg-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
            uux=dux/Lu;uuz=duz/Lu
            ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
            iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz));il=ix*iz
            d2x=xg-xc;d2y=yg;d2z=zg-zc;Ru=torch.sqrt(d2x**2+d2y**2+d2z**2)
            t1=d2x/Ru;t2=d2z/Ru;ax=(1-il)*(kx-t1);az=(1-il)*(kz-t2)
            sx=torch.sinc((math.pi/lam)*Lx*ax);sz=torch.sinc((math.pi/lam)*Lz*az)
            p1=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(th)
            a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
            v1m=v1c/Rw;v1p=v1pc*Rw;v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p))
            H1=v1*a1.conj()
            p2=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
            a2=oE*torch.complex(torch.cos(p2),torch.sin(p2))
            v2m=eta*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(ps),v2m*torch.sin(ps))
            H2=v2.unsqueeze(1)*a2.conj()
            Ht=math.sqrt(E/Lp)*(H1+H2)
            fm=(Lx*Lz*sx*sz)/(lam*Ru)
            fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
            fc=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp))
            He=Ht*fc.unsqueeze(1);Hw=torch.sum(He,dim=1)/math.sqrt(E)
            dp=(torch.abs(Hw)**2)*pm*PB;it=(nu-1)*dp;sr=dp/(it+N0f)
            ab=math.pi/3.0;Ses=torch.zeros(Ng,device=device)
            rn=torch.sqrt((xg-xc)**2+yg**2+(zg-zc)**2+1e-12)
            for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
                dot=torch.abs((xg-xc)*math.cos(a)+yg*math.sin(a))
                cp=dot/rn;ph_b=torch.acos(torch.clamp(cp,0,1))
                Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi
                Ses=Ses+torch.clamp(Se,1/3,1)
            sr_se=((Ses/4)*dp)/((Ses/4)*it+N0f)
            bits=(torch.log2(1+sr_se)<Rth).float()
            bo[j]=torch.sum(bits*gw)
        out.append(bo.cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

print('Physics engine ready.')

# ============================================================
# 3. Set Room Dimensions (EDIT THESE)
# ============================================================
ROOM_X, ROOM_Y = 12.0, 8.0  # <-- CHANGE TO YOUR ROOM SIZE
print(f'\nRoom: {ROOM_X}m × {ROOM_Y}m')

# ============================================================
# 4. Phase 1: NSGA-II on LGBM Surrogate
# ============================================================
class SurrogateProblem(ElementwiseProblem):
    def __init__(self, rx, ry):
        self.rx, self.ry = rx, ry
        lb=np.array([0.0,0.0,0.05,0.05])
        ub=np.array([rx,rz, rx, rz])
        super().__init__(n_var=4, n_obj=2, n_ieq_constr=4, xl=lb, xu=ub)
    def _evaluate(self, x, out, *a, **k):
        xc,zc,Lx,Lz = x
        out["G"] = [Lx/2-xc, xc+Lx/2-self.rx, Lz/2-zc, zc+Lz/2-rz]
        pred = float(lgbm_predict(x.reshape(1,-1), self.rx, self.ry)[0])
        out["F"] = [Lx*Lz, pred]

print('\nPhase 1: NSGA-II on LGBM surrogate (200 gen)...')
t0 = time.time()
algo_s = NSGA2(pop_size=300, n_offsprings=150, sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15), mutation=PM(prob=0.25, eta=20), eliminate_duplicates=True)
res_s = minimize(SurrogateProblem(ROOM_X, ROOM_Y), algo_s, get_termination('n_gen', 200), seed=42, verbose=True)
t_phase1 = time.time()-t0
print(f'Phase 1 done: {t_phase1:.0f}s, {len(res_s.F)} pts')

# ============================================================
# 5. Phase 2: Physics Warm-Start Refinement
# ============================================================
class PhysicsProblem(ElementwiseProblem):
    def __init__(self, rx, ry):
        self.rx, self.ry = rx, ry
        lb=np.array([0.0,0.0,0.05,0.05])
        ub=np.array([rx,rz, rx, rz])
        super().__init__(n_var=4, n_obj=2, n_ieq_constr=4, xl=lb, xu=ub)
    def _evaluate(self, x, out, *a, **k):
        xc,zc,Lx,Lz = x
        out["G"] = [Lx/2-xc, xc+Lx/2-self.rx, Lz/2-zc, zc+Lz/2-rz]
        out["F"] = [Lx*Lz, float(physics_outage(x.reshape(1,-1), self.rx, self.ry)[0])]

# Build warm-start population
F_s = res_s.F; mask = F_s[:,1] < 0.30
warm_X = res_s.X[mask][:250]
np.random.seed(99)
lb_r=np.array([0.0,0.0,0.05,0.05]); ub_r=np.array([ROOM_X,rz,ROOM_X,rz])
rand_X = np.random.uniform(low=lb_r, high=ub_r, size=(50,4))
hybrid_X = np.vstack([warm_X, rand_X])
warm_pop = Population.new("X", hybrid_X)

print(f'\nPhase 2: Physics warm-start ({len(warm_X)} elites + 50 random) → AGE-MOEA 20 gen...')
algo_p = AGEMOEA(pop_size=300, sampling=warm_pop,
    crossover=SBX(prob=0.9, eta=15), mutation=PM(prob=0.35, eta=15))
res_p = minimize(PhysicsProblem(ROOM_X, ROOM_Y), algo_p, get_termination('n_gen', 20), seed=42, verbose=True)
t_total = time.time()-t0
print(f'Phase 2 done: {t_total-t_phase1:.0f}s. Total: {t_total:.0f}s')

# ============================================================
# 6. Results
# ============================================================
F_final = res_p.F
feas = F_final[:,1] <= 0.10
if feas.any():
    bi = np.argmin(F_final[feas,0])
    ba, bo = F_final[feas,0][bi], F_final[feas,1][bi]
    bx = res_p.X[np.where(feas)[0][bi]]
    print(f'\nKnee: [{bx[0]:.3f},{bx[1]:.3f},{bx[2]:.3f},{bx[3]:.3f}] area={ba:.4f} m² outage={bo*100:.2f}%')

# Plot
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5.5))
idx_s=np.argsort(res_s.F[:,1]);Fs=res_s.F[idx_s]
idx_p=np.argsort(F_final[:,1]);Fp=F_final[idx_p]
ax1.plot(Fs[:,1]*100,Fs[:,0],'o-',color='darkorange',markersize=2,linewidth=1,label='LGBM Surrogate')
ax1.plot(Fp[:,1]*100,Fp[:,0],'D-',color='darkgreen',markersize=3,linewidth=1.5,label='Physics Refined')
ax1.axvline(x=10,color='red',ls='--',alpha=0.5);ax1.legend();ax1.grid(alpha=0.3)
ax1.set_xlabel('Outage [%]');ax1.set_ylabel('Area [m²]')
ax1.set_title(f'Room {ROOM_X}×{ROOM_Y}m: Pareto Front')
if feas.any(): ax1.plot(bo*100,ba,'r*',markersize=14,label=f'Knee: {ba:.4f} m²')

# Wall canvas
bx_f=np.array(bx) if feas.any() else np.array([ROOM_X/2,1.5,ROOM_X*0.9,0.2])
from matplotlib.patches import Rectangle
rect=Rectangle((bx_f[0]-bx_f[2]/2,bx_f[1]-bx_f[3]/2),bx_f[2],bx_f[3],fill=True,color='darkgreen',alpha=0.5,edgecolor='black',linewidth=2)
ax2.add_patch(rect);ax2.plot(bx_f[0],bx_f[1],'k*',markersize=12)
ax2.set_xlim(0,ROOM_X);ax2.set_ylim(0,rz);ax2.set_aspect('equal')
ax2.set_xlabel('x [m]');ax2.set_ylabel('z [m]')
ax2.set_title(f'Knee Window on {ROOM_X}×{ROOM_Y}m Wall');ax2.grid(alpha=0.3)
plt.tight_layout();plt.savefig('dynamic_pareto.png',dpi=150);plt.show()

from IPython.display import FileLink,display
display(FileLink('dynamic_pareto.png'))